# Lab 6.6 &mdash; Your First RAG Chain

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Wire a retriever, a prompt, the model and a parser into one LCEL chain
- Work out what <code>RunnablePassthrough</code> is actually for
- Write the refusal clause &mdash; and measure what happens without it
- Ask the corpus something it does not contain, and watch which version makes something up

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a
> `Document`, a Chroma collection, a retriever, a chain), so they are deterministic and do not
> depend on the chat model. Cells marked **Run it for real** put your code in front of the
> sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **Two different models are in play, and only one of them is billed.** The **chat model**
> (`qwen36-35b-a3b-lab`) answers questions and is reached over the gateway. The **embedding
> model** (`all-MiniLM-L6-v2`, 384 dimensions) turns text into vectors and runs on this pod's
> own CPU &mdash; no key, no gateway, no tokens. Keeping them straight is most of Module 6.

> **This lab calls the chat model.** Everything up to the last cell of each section
> is still offline: a chain is an object, and building it needs no gateway.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap, warnings
from typing import Any, Callable

warnings.filterwarnings("ignore")     # sentence-transformers is chatty on first import

WORK = os.path.join("/tmp", "awmas-lab-6-06")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the CHAT model: qwen, through the sandbox gateway -------------------
# Already configured -- nothing to install, no key to register. Read from the
# environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Chat model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- the EMBEDDING model: local, free, nothing to configure --------------
# all-MiniLM-L6-v2, 384 dimensions. It runs on this pod's CPU and has nothing to do with
# the chat model above: no gateway, no key, no tokens billed. The cache is already warm
# in your sandbox, so the first call is a second or two, not a download.
#
# It is reached through onnxruntime rather than torch, and that is a measured choice
# rather than a taste: same model, same vectors, ~170 MB of memory instead of ~840. Your
# whole sandbox has 2.5 GB for every notebook you leave open, and a kernel you have
# forgotten about is still holding its share.
from langchain_core.embeddings import Embeddings

class MiniLMEmbeddings(Embeddings):
    """all-MiniLM-L6-v2 behind LangChain's Embeddings interface.

    Two methods is the whole contract -- which is why a store, a splitter and a chain
    never need to know which model is underneath, or what runtime it uses."""

    def __init__(self):
        from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2
        self._fn = ONNXMiniLM_L6_V2()

    def embed_documents(self, texts: list) -> list:
        return [[float(x) for x in v] for v in self._fn(list(texts))]

    def embed_query(self, text: str) -> list:
        return [float(x) for x in self._fn([text])[0]]


_emb_cache = {}
def get_embeddings():
    """The embedding model, built once per kernel."""
    if "model" not in _emb_cache:
        _emb_cache["model"] = MiniLMEmbeddings()
    return _emb_cache["model"]

print("work dir   :", WORK)
print("chat model :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Ten short passages from a company handbook. Note what each one carries besides its text:
# a category, a source file and a page. Those three are what Lab 6.3 filters on and what
# Lab 6.7 cites -- metadata is not decoration, it is the half of retrieval that is exact.
#
# Note also what is NOT here: nothing mentions salary, notice period or the share price.
# Labs 6.6 and 6.8 need that gap, because refusing is a feature.

HANDBOOK = [
    {"text": "Annual leave is 24 days per year for full-time employees. Leave must be applied "
             "for at least 3 working days in advance. Unused annual leave cannot be carried "
             "forward to the next financial year.",
     "category": "leave", "source": "handbook.pdf", "page": 5},
    {"text": "Sick leave is 12 days per year. Notify your manager by 10 AM on the day of "
             "absence. A medical certificate is required for absences of more than 2 "
             "consecutive days.",
     "category": "leave", "source": "handbook.pdf", "page": 5},
    {"text": "Maternity leave is 26 weeks of paid leave. Paternity leave is 2 weeks. Both "
             "must be applied for at least 30 days before the expected date.",
     "category": "leave", "source": "handbook.pdf", "page": 6},
    {"text": "Employees may work from home up to 3 days per week with team lead approval. "
             "Core hours are 10 AM to 4 PM IST, and you must be reachable during them.",
     "category": "wfh", "source": "handbook.pdf", "page": 8},
    {"text": "A VPN connection is mandatory for reaching internal systems from home. "
             "Contact the IT helpdesk for VPN setup.",
     "category": "wfh", "source": "handbook.pdf", "page": 8},
    {"text": "Internet reimbursement is 1,500 per month for employees working from home. "
             "Submit the broadband bill to finance by the 5th of each month.",
     "category": "expense", "source": "handbook.pdf", "page": 9},
    {"text": "Travel expenses must be submitted with original receipts within 7 working days "
             "of travel. The meal allowance during client visits is 500 per day.",
     "category": "expense", "source": "handbook.pdf", "page": 12},
    {"text": "Laptops are provided by the company and replaced every 3 years. Software "
             "licence requests go through the IT helpdesk and must not be bought directly.",
     "category": "tech", "source": "tech-guide.pdf", "page": 7},
    {"text": "The backend stack is Python with FastAPI, and Java with Spring Boot. New "
             "services should use Python unless there is a specific reason not to. "
             "PostgreSQL is the primary database.",
     "category": "tech", "source": "tech-guide.pdf", "page": 3},
    {"text": "The Bangalore office is the headquarters, on the 5th floor, with 200+ staff. "
             "The Mumbai office is in the Worli business district, Tower A, 12th floor.",
     "category": "office", "source": "office-directory.pdf", "page": 15},
]

print(f"{len(HANDBOOK)} passages, "
      f"{len({d['category'] for d in HANDBOOK})} categories, "
      f"{len({d['source'] for d in HANDBOOK})} source files")

In [ ]:
# ------------------------------------------------- the corpus as LangChain Documents
from langchain_core.documents import Document

def handbook_documents() -> list:
    """One Document per passage: the text, and everything else as metadata."""
    return [Document(page_content=d["text"],
                     metadata={"category": d["category"],
                               "source": d["source"],
                               "page": d["page"]})
            for d in HANDBOOK]

print(len(handbook_documents()), "Document objects")

In [ ]:
# ------------------------------------------------- carried forward from Lab 6.5 (given)
from langchain_chroma import Chroma

_store = {}
def store():
    if "s" not in _store:
        _store["s"] = Chroma.from_documents(documents=handbook_documents(),
                                            embedding=get_embeddings(),
                                            collection_name="handbook_rag")
    return _store["s"]

def retriever(k: int = 3):
    return store().as_retriever(search_kwargs={"k": k})

print("store and retriever ready")

## Concept

The chain is one pipe with a dictionary at the front:

```
{"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
```

Invoke it with a string. That one string goes to **both** keys: down the left branch it hits
the retriever and comes back as chunks; down the right branch `RunnablePassthrough` hands it
straight to the template. The prompt then has both `{context}` and `{question}` to fill.

Take the passthrough out and the template has nothing to put in `{question}`. That is all it
is for, and it is the piece people cannot explain in interviews.

## Section 1 &mdash; The chain

`format_docs` is given &mdash; joining strings is not the lesson. The decision is what fills
`"question"`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs) -> str:
    """Given -- Documents to one string for the prompt."""
    return "\n\n".join(d.page_content for d in docs)


def question_branch():
    """The chain is invoked with a plain string. The left branch sends it to the retriever.
    What does the right branch need, so the template's {question} is the ORIGINAL string?"""
    return RunnablePassthrough()


def build_chain(prompt):
    """Given -- your branch, wired in."""
    return ({"context": retriever() | format_docs, "question": question_branch()}
            | prompt | get_llm() | StrOutputParser())

In [ ]:
# --- Self-check: Section 1   (the branch and the formatter -- the chain needs a gateway)
check("the question branch is a Runnable",
      lambda: hasattr(question_branch(), "invoke"))
check("it passes a string through UNCHANGED",
      lambda: question_branch().invoke("How many days of leave?") == "How many days of leave?",
      "anything that transforms the question here and the template gets the wrong text")
check("format_docs turns retrieved Documents into one string",
      lambda: isinstance(format_docs(retriever().invoke("annual leave")), str))
check("and the retrieved text really is in it",
      lambda: "24 days" in format_docs(retriever().invoke("How many days off do I get?")))

## Section 2 &mdash; The refusal clause

The corpus says nothing about salaries, notice periods or the share price. Ask about them
anyway.

A prompt without an explicit instruction to refuse will get an answer invented out of
whatever chunks happened to come back &mdash; because three chunks always come back. The
clause that fixes it is one sentence, and it is the difference between a demo and something
you would put in front of staff.

In [ ]:
def refusal_clause() -> str:
    """One sentence telling the model what to do when the context does not contain the
    answer. It has to name the behaviour you want, not just discourage the one you don't."""
    return ("If the context does not contain the answer, reply exactly "
            "\"I don't have that information in the handbook.\" and nothing else.")


LOOSE = ChatPromptTemplate.from_template(
    "Answer the question using the context below.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:")

def strict_prompt():
    """Given -- the same prompt, plus your clause."""
    return ChatPromptTemplate.from_template(
        "Answer the question using ONLY the context below. " + refusal_clause() +
        "\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:")

In [ ]:
# --- Self-check: Section 2   (prompt objects -- no call yet)
check("your clause is a sentence, not a word",
      lambda: len(refusal_clause().split()) >= 6)
check("it says what to do, not only what to avoid",
      lambda: any(w in refusal_clause().lower()
                  for w in ("say", "reply", "respond", "answer", "state")),
      "'do not make things up' tells the model what not to write, not what to write instead")
check("the strict prompt still has both variables",
      lambda: set(strict_prompt().input_variables) == {"context", "question"})
check("and the clause really is in the rendered prompt",
      lambda: refusal_clause()[:24] in
              strict_prompt().format(context="c", question="q"))
check("the loose prompt does NOT contain it",
      lambda: refusal_clause()[:24] not in LOOSE.format(context="c", question="q"))

In [ ]:
# --- Run it for real -------------------------------------------------------
# Four questions. Three the handbook answers, one it does not.
IN_SCOPE = ["How many days of annual leave do I get?",
            "Can I work from home, and how often?",
            "What is the meal allowance on a client visit?"]
OUT_OF_SCOPE = "What is the company's share price?"

def answers():
    chain = build_chain(strict_prompt())
    for q in IN_SCOPE:
        print(f"Q: {q}\nA: {chain.invoke(q).strip()}\n")

if llm_ready():
    guard(answers)

In [ ]:
# --- Run it for real: the clause, measured ---------------------------------
# The same out-of-scope question through both prompts. Three handbook chunks are
# retrieved either way -- none of them about the share price.
def refusal_matters():
    for label, prompt in (("no clause", LOOSE), ("with clause", strict_prompt())):
        out = build_chain(prompt).invoke(OUT_OF_SCOPE).strip()
        print(f"  [{label:11}] {out[:150]}")
    print("\n  retrieved for that question:")
    for d in retriever().invoke(OUT_OF_SCOPE):
        print(f"    ({d.metadata['category']}) {d.page_content[:56]}...")

if llm_ready():
    guard(refusal_matters)

In [ ]:
score()

## Your turn

1. Delete `"question": question_branch()` from the dictionary and invoke the chain. Read the
   error carefully &mdash; it names exactly what the passthrough was doing.
2. Run the out-of-scope question through the loose prompt five times. Count how many answers
   invent a number. One run is an anecdote; five is the beginning of an eval set, and
   Module 7 turns it into one.
3. Your clause asks for an exact sentence. Now write a version that also says *which*
   category it searched, so the user learns something from the refusal. Does the model
   comply reliably? Instructions that require the model to report on its own retrieval are
   a common source of confident nonsense.